<a href="https://colab.research.google.com/github/aabadmo4/DGT_Incidencias/blob/main/Boletin_ZGZ%2BTrafico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Boletín de tráfico DGT + calles de Zaragoza — explicado línea a línea

Este notebook reproduce y comenta la versión actual de `boletin_estado_carreteras.py`: reintentos ante saturación de Gemini, detalle ampliado para Zaragoza provincia (sentido, carril, punto kilométrico) en los datos de la DGT, y ahora también un bloque al principio del boletín con las incidencias de **calles de la ciudad de Zaragoza** (cortes de tráfico, obras, desvíos) publicadas por el Ayuntamiento.

Estructura del notebook:
1. Instalación de dependencias
2. Credenciales
3. Funciones de fecha/hora y saludo
4. Extracción de sentido, carril y punto kilométrico (DGT)
5. Extracción y limpieza de incidencias DGT (con detalle extra para Zaragoza)
6. Incidencias de calles de Zaragoza (Ayuntamiento)
7. **Celda de diagnóstico**: probar solo la API del Ayuntamiento y ver el JSON crudo
8. Generación de audio (TTS)
9. Llamada a Gemini con reintentos ante error 503
10. Envío a Telegram
11. Función que genera el boletín completo (texto + audio) sin enviarlo
12. Celda para **generar** el boletín
13. Celda para **escuchar/leer** el resultado antes de enviarlo
14. Celda para **enviar manualmente** a Telegram

## 1. Instalación de dependencias

`lxml` para parsear el XML de la DGT, `google-genai` para llamar a Gemini, `pydub` para acelerar el audio (necesita `ffmpeg` instalado en el sistema, por eso el `apt-get`).

In [13]:
!apt-get -qq install -y ffmpeg
!pip install -q requests lxml pydub google-genai

## 2. Imports

- `re`: expresiones regulares para limpiar textos.
- `time`: para las esperas entre reintentos a Gemini.
- `json`: solo para imprimir bonito el JSON crudo en la celda de diagnóstico.
- `datetime` + `zoneinfo`: fecha/hora, usando la zona horaria real de Madrid.
- `requests`: llamadas HTTP (a la DGT, al Ayuntamiento de Zaragoza, a Google Translate TTS y a Telegram).
- `defaultdict`: para agrupar incidencias por región sin comprobar antes si la clave existe.
- `etree` de `lxml`: parsear el XML DATEX II de la DGT.
- `genai` y `genai_errors`: cliente de la API de Gemini y sus excepciones (para detectar el error 503).
- `AudioSegment` de `pydub`: manipular el audio generado (acelerarlo).

In [14]:
import re
import time
import json
import datetime
from zoneinfo import ZoneInfo
import requests
from collections import defaultdict
from lxml import etree
from google import genai
from google.genai import errors as genai_errors
from pydub import AudioSegment
from IPython.display import Audio, display

## 3. Credenciales

En el script original se leen de variables de entorno (`os.environ.get(...)`), como las inyecta la GitHub Action mediante *secrets*. Aquí, para ejecutarlo a mano en Colab, las pedimos con `getpass` (no se muestran en pantalla ni quedan escritas en el notebook).

Si prefieres no teclearlas cada vez, guárdalas como *Secrets* de Colab (icono de llave 🔑 en el panel izquierdo) y sustituye esta celda por `from google.colab import userdata; GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')`, etc.

**Nota:** para la celda de diagnóstico de la sección 7 no hace falta ninguna credencial — la API del Ayuntamiento es pública.

In [16]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
TELEGRAM_BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
TELEGRAM_CHAT_ID = userdata.get('TELEGRAM_CHAT_ID')

## 4. Fecha, hora y saludo

- `DIAS_ES` / `MESES_ES`: traducen día de la semana y mes a español (Python no lo hace de forma nativa sin configurar el *locale* del sistema).
- `obtener_hora_madrid()`: hora actual ya localizada en `Europe/Madrid` (con `ZoneInfo`, corrige automáticamente el cambio de hora de invierno/verano).
- `obtener_fecha_hora_generacion()`: construye la frase tipo *"sábado 19 de septiembre, 10:15 horas"* que se antepone al boletín.
- `obtener_saludo_y_momento()`: según la hora de Madrid, decide "Buenos días", "Buenas tardes" o "Buenas noches".

In [17]:
DIAS_ES = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
MESES_ES = [
    "enero", "febrero", "marzo", "abril", "mayo", "junio",
    "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
]


def obtener_hora_madrid():
    return datetime.datetime.now(ZoneInfo("Europe/Madrid"))


def obtener_fecha_hora_generacion():
    ahora = obtener_hora_madrid()
    dia_semana = DIAS_ES[ahora.weekday()]
    mes = MESES_ES[ahora.month - 1]
    return f"{dia_semana} {ahora.day} de {mes}, {ahora.hour:02d}:{ahora.minute:02d} horas"


def obtener_saludo_y_momento():
    hora = obtener_hora_madrid().hour
    if 6 <= hora < 12:
        return "Buenos días"
    elif 12 <= hora < 20:
        return "Buenas tardes"
    else:
        return "Buenas noches"

## 5. Sentido, carril y punto kilométrico (DGT)

Consultando la especificación DATEX II de la DGT, estos datos vienen en etiquetas concretas del XML: `tpegDirection` (rumbo cardinal), `tpegDirectionRoad` (sentido de la kilometración: `positive`/`negative`/`both`), `laneUsage` (carril afectado) y `kilometerPoint`.

`extraer_sentido_carril_pk(record)` busca esas etiquetas concretas con xpath (`local-name()` porque el XML usa varios namespaces) en vez de intentar adivinarlas del texto suelto del registro, que es frágil.

- Traduce los rumbos cardinales y los valores de sentido/carril a español.
- Si hay más de un punto kilométrico (inicio y fin de un tramo), devuelve el rango; si solo hay uno, ese punto.
- Devuelve `(sentido, carriles, pk_str)`, cada uno `None`/lista vacía si no hay dato.

In [18]:
def extraer_sentido_carril_pk(record):
    direcciones_cardinales = {
        "NORTHBOUND": "sentido norte", "SOUTHBOUND": "sentido sur",
        "EASTBOUND": "sentido este", "WESTBOUND": "sentido oeste",
        "NORTHEASTBOUND": "sentido noreste", "NORTHWESTBOUND": "sentido noroeste",
        "SOUTHEASTBOUND": "sentido sureste", "SOUTHWESTBOUND": "sentido suroeste",
    }
    sentidos_kilometracion = {
        "POSITIVE": "sentido creciente", "NEGATIVE": "sentido decreciente", "BOTH": "ambos sentidos",
    }
    carriles_traducidos = {
        "RIGHTLANE": "carril derecho", "LEFTLANE": "carril izquierdo", "CENTRALLANE": "carril central",
        "HARDSHOULDER": "arcén", "SHOULDERLANE": "arcén", "BUSLANE": "carril bus",
        "ALLLANESCOMPLETECARRIAGEWAY": "todos los carriles",
    }

    partes_sentido = []
    for tag in ("tpegDirection", "tpegDirectionRoad"):
        for v in record.xpath(f'.//*[local-name()="{tag}"]/text()'):
            v_upper = v.strip().upper()
            if v_upper in direcciones_cardinales and direcciones_cardinales[v_upper] not in partes_sentido:
                partes_sentido.append(direcciones_cardinales[v_upper])
            elif v_upper in sentidos_kilometracion and sentidos_kilometracion[v_upper] not in partes_sentido:
                partes_sentido.append(sentidos_kilometracion[v_upper])
    sentido = " / ".join(partes_sentido) if partes_sentido else None

    carriles = []
    for v in record.xpath('.//*[local-name()="laneUsage"]/text()'):
        v_upper = v.strip().upper()
        if v_upper in carriles_traducidos and carriles_traducidos[v_upper] not in carriles:
            carriles.append(carriles_traducidos[v_upper])

    puntos_km = sorted(set(record.xpath('.//*[local-name()="kilometerPoint"]/text()')))
    if not puntos_km:
        pk_str = None
    elif len(puntos_km) == 1:
        pk_str = f"pk {puntos_km[0]}"
    else:
        pk_str = f"entre pk {puntos_km[0]} y pk {puntos_km[-1]}"

    return sentido, carriles, pk_str

## 6. Extracción y limpieza de incidencias DGT

### `limpiar_y_extraer_detalles(record)`
Recibe un `situationRecord` (un bloque XML con una incidencia) y saca de él algo legible:

- `traducciones_causa`: traduce los códigos DATEX II a español, incluyendo incidencias leves como `TRAFFICCONGESTION` → "Retención" y `OBSTRUCTION` → "Obstáculo en la vía".
- `valores_estructurales`: conjunto de valores (sentido, carril, `unknown`...) que antes se colaban como si fueran nombres de municipio; ahora se descartan aquí explícitamente, igual que cualquier texto que termine en `bound`.
- El bucle separa: causas conocidas, provincia (Teruel/Zaragoza/Huesca/Navarra/La Rioja) y municipios.
- Llama a `extraer_sentido_carril_pk` para el detalle estructurado.
- **Si la provincia es "Zaragoza"** y hay detalle disponible, lo añade: PK, sentido y carril(es). El resto de provincias se queda con el resumen básico.

### `obtener_incidencias_texto()`
Descarga el XML de la DGT, recorre cada `situationRecord`, comprueba si menciona Aragón/Navarra/La Rioja y arma el texto por regiones que luego lee la IA.

In [20]:
def limpiar_y_extraer_detalles(record):
    traducciones_causa = {
        "ROADWORKS": "Obras", "ROADMAINTENANCE": "Mantenimiento",
        "CARRIAGEWAYCLOSURE": "Corte total de calzada", "ACCIDENT": "Accidente",
        "POORWEATHERCONDITIONS": "Meteorología adversa", "SNOW": "Nieve",
        "ICE": "Hielo", "FLOODING": "Inundación", "OBSTRUCTION": "Obstáculo en la vía",
        "TRAFFICCONGESTION": "Retención"
    }
    valores_estructurales = {
        'unspecifiedcarriageway', 'unknown', 'both', 'negative', 'positive',
        'rightlane', 'leftlane', 'centrallane', 'hardshoulder', 'shoulderlane',
        'buslane', 'alllanescompletecarriageway'
    }

    raw_texts = [t.strip() for t in record.xpath('.//text()') if t.strip()]
    municipios, provincia, causas = [], "", []

    for t in raw_texts:
        t_clean = t.strip()
        if re.match(r'^\d{4}-\d{2}-\d{2}', t_clean) or re.match(r'^-?\d+\.\d+$', t_clean) or re.match(r'^[A-Z0-9_]{8,}$', t_clean):
            continue
        t_lower = t_clean.lower()
        if t_lower in ['true', 'false', 'dgt', 'certain', 'active', 'segment', 'mandatory', 'anyvehicle']:
            continue
        if t_lower in valores_estructurales or t_lower.endswith('bound'):
            continue

        t_upper = t_clean.upper()
        if t_upper in traducciones_causa:
            causas.append(traducciones_causa[t_upper])
            continue

        if t_clean in ["Teruel", "Zaragoza", "Huesca", "Navarra", "La Rioja"]:
            provincia = t_clean
        elif len(t_clean) > 2 and not t_clean.replace('.', '').isdigit() and t_clean not in ["Aragón", "Comunidad Foral de Navarra"]:
            if t_clean not in municipios:
                municipios.append(t_clean)

    sentido, carriles, pk_str = extraer_sentido_carril_pk(record)

    ubicacion_str = f"Entre/En: {', '.join(municipios)}" if municipios else "Tramo local"
    causa_str = f"Incidencia: {', '.join(set(causas))}" if causas else "Afección en la vía"

    detalle_extra = []
    if pk_str:
        detalle_extra.append(pk_str.capitalize())
    if sentido:
        detalle_extra.append(f"Sentido: {sentido}")
    if carriles:
        detalle_extra.append(f"Carril(es): {', '.join(carriles)}")

    if provincia == "Zaragoza" and detalle_extra:
        return f"{provincia} | {ubicacion_str} | {causa_str} | " + " | ".join(detalle_extra)
    return f"{provincia} | {ubicacion_str} | {causa_str}"


def obtener_incidencias_texto():
    url = "https://nap.dgt.es/datex2/v3/dgt/SituationPublication/datex2_v37.xml"
    headers = {'User-Agent': 'Mozilla/5.0'}

    regiones_mapa = {
        "ARAGÓN": ["ZARAGOZA", "HUESCA", "TERUEL", "ARAGON", "ARAGÓN"],
        "COMUNIDAD FORAL DE NAVARRA": ["NAVARRA", "PAMPLONA"],
        "LA RIOJA": ["RIOJA", "LOGROÑO"]
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()

        parser = etree.XMLParser(recover=True, encoding='utf-8')
        root = etree.fromstring(response.content, parser=parser)

        incidencias_por_zona = defaultdict(list)

        for record in root.xpath('//*[local-name()="situationRecord"]'):
            roads = record.xpath('.//*[local-name()="roadName"]/text()')
            road_name = roads[0].strip() if roads else "Vía local"

            raw_texts = [t.strip() for t in record.xpath('.//text()') if t.strip()]
            texto_evaluacion = f"{road_name} " + " ".join(raw_texts).upper()

            region_encontrada = None
            for region, terminos in regiones_mapa.items():
                if any(term in texto_evaluacion for term in terminos):
                    region_encontrada = region
                    break

            if region_encontrada:
                resumen = limpiar_y_extraer_detalles(record)
                incidencias_por_zona[region_encontrada].append(f"- Código oficial: {road_name} -> {resumen}")

        texto_resultado = ""
        for reg in ["ARAGÓN", "COMUNIDAD FORAL DE NAVARRA", "LA RIOJA"]:
            if reg in incidencias_por_zona:
                texto_resultado += f"\n--- REGIÓN: {reg} ---\n" + "\n".join(incidencias_por_zona[reg]) + "\n"

        return texto_resultado if texto_resultado else "Sin incidencias."

    except Exception as e:
        return f"Error extrayendo datos: {e}"

## 7. Incidencias de calles de Zaragoza (Ayuntamiento)

Esto es nuevo: el Ayuntamiento de Zaragoza publica un dataset abierto llamado **"Incidencias en la Vía Pública"** con tres tipos: `0` Cortes de Agua, `1` Cortes de Tráfico, `2` Afecciones Importantes (obras, desvíos). La DGT solo cubre carreteras interurbanas, así que esto añade el detalle a nivel de calle dentro de la ciudad.

`obtener_incidencias_calles_zaragoza()`:
- Pide al endpoint `tipo.id==1,tipo.id==2` (excluye cortes de agua).
- Trata la respuesta de forma defensiva: si viene como lista la usa tal cual, si viene como diccionario prueba varias claves habituales (`incidencia`, `results`, `items`).
- Para cada incidencia intenta leer `calle`/`title`, `tramo`, `motivo`/`description`, `inicio` y `fin`, con valores por defecto si algún campo no existe.
- **Importante:** el formato exacto de esta API no se ha podido verificar en vivo todavía. Por eso existe la celda de diagnóstico justo debajo (sección 7b): sirve para ver el JSON tal cual lo devuelve el Ayuntamiento y ajustar los nombres de campo en esta función si hiciera falta.

In [27]:
def obtener_incidencias_calles_zaragoza():
    # URL corregida: via-publica va ANTES de incidencia
    url = "https://www.zaragoza.es/sede/servicio/via-publica/incidencia.json"
    params = {"srsname": "utm30n", "rows": 30, "q": "tipo.id==1,tipo.id==2"}
    headers = {
        'User-Agent': 'Mozilla/5.0',
        'Accept': 'application/json'
    }

    try:
        response = requests.get(url, params=params, headers=headers, timeout=20)
        response.raise_for_status()
        data = response.json()

        if isinstance(data, list):
            registros = data
        elif isinstance(data, dict):
            registros = data.get("result") or data.get("incidencia") or data.get("results") or data.get("items") or []
        else:
            registros = []

        lineas = []
        for item in registros:
            if not isinstance(item, dict):
                continue

            calle = item.get("calle") or item.get("title") or item.get("nombre") or "Vía no especificada"
            tramo = item.get("tramo") or ""
            motivo = item.get("motivo") or item.get("description") or "Obras/afección en la vía"
            inicio = item.get("inicio") or ""
            fin = item.get("fin") or ""

            # Limpiar la marca de tiempo 'T00:00:00' si viene en los strings
            if fin and 'T' in str(fin):
                fin = str(fin).split('T')[0]
            if inicio and 'T' in str(inicio):
                inicio = str(inicio).split('T')[0]

            partes = [str(calle)]
            if tramo:
                partes.append(str(tramo))
            partes.append(str(motivo))
            if inicio or fin:
                partes.append(f"(hasta {fin})" if fin else f"(desde {inicio})")

            lineas.append("- " + " | ".join(p for p in partes if p))

        return "\n".join(lineas) if lineas else "Sin incidencias destacadas en la ciudad."

    except Exception as e:
        return f"No se pudo consultar incidencias municipales de Zaragoza: {e}"

### 7b. Celda de diagnóstico — SOLO la API del Ayuntamiento

Ejecuta esta celda de forma independiente (no depende de credenciales ni de nada anterior salvo los imports de la sección 2) para ver:
1. El código de estado HTTP de la petición.
2. La URL final exacta que se llamó.
3. El JSON crudo, bonito, tal cual lo devuelve el Ayuntamiento (o el texto de la respuesta si no es JSON válido).
4. El resultado ya parseado por `obtener_incidencias_calles_zaragoza()`, para comparar.

Si el JSON real trae los campos con otro nombre, aquí los verás y podrás decirme cuáles son para ajustar la función de la sección 7.

In [36]:
# URL corregida: via-publica va ANTES de incidencia
url_diagnostico = "https://www.zaragoza.es/sede/servicio/via-publica/incidencia.json"
params_diagnostico = {"srsname": "utm30n", "rows": 30, "q": "tipo.id==1,tipo.id==2"}
headers_diagnostico = {
    'User-Agent': 'Mozilla/5.0',
    'Accept': 'application/json'
}

resp = requests.get(url_diagnostico, params=params_diagnostico, headers=headers_diagnostico, timeout=20)

print("Código de estado HTTP:", resp.status_code)
print("URL final:", resp.url)
print("Content-Type:", resp.headers.get("Content-Type"))
print("-" * 60)

try:
    data_diagnostico = resp.json()
    print("JSON recibido (bonito):")
    print(json.dumps(data_diagnostico, indent=2, ensure_ascii=False)[:5000])
except ValueError:
    print("La respuesta NO es JSON válido. Texto crudo (primeros 3000 caracteres):")
    print(resp.text[:3000])

print("-" * 60)
print("Resultado de obtener_incidencias_calles_zaragoza():")
print(obtener_incidencias_calles_zaragoza())

Código de estado HTTP: 200
URL final: https://www.zaragoza.es/sede/servicio/via-publica/incidencia.json?srsname=utm30n&rows=30&q=tipo.id%3D%3D1%2Ctipo.id%3D%3D2
Content-Type: application/json;charset=UTF-8
------------------------------------------------------------
JSON recibido (bonito):
{
  "totalCount": 57,
  "start": 0,
  "rows": 30,
  "result": [
    {
      "id": 25688,
      "title": "DR. BLANCO CORDERO Y PEDRO DE ALVARADO",
      "calle": "BLANCO CORDERO (DR.), 11",
      "motivo": "Para la reforma de las calles y de las viviendas es necesario el corte de las calles",
      "tramo": "Dr. Blanco Cordero se corta en el tramo comprendido entre el n.o 14 y la calle López Abadía.",
      "observaciones": "se mantienen cortadas las calles Pedro de Alvarado en su totalidad y Blanco Cordero en el tramo comprendido entre el n.º 14 y la calle López Abadía quedando en fondo de saco y doble sentido de circulación entre el n.o 9 y el n.o 14 y manteniendo prohibido el estacionamiento 15 met

## 8. Generación de audio (texto a voz)

- `acelerar_audio(...)`: carga el mp3 con `pydub`, sube el `frame_rate` multiplicándolo por `velocidad` y lo "reetiqueta" al frame rate original — así suena más rápido.
- `generar_voz_espanol(...)`: limpia asteriscos/almohadillas, trocea el texto en frases (y subdivide las muy largas, por el límite del endpoint de TTS), pide el audio fragmento a fragmento, concatena y acelera.

In [37]:
def acelerar_audio(archivo_entrada, archivo_salida, velocidad=1.25):
    audio = AudioSegment.from_file(archivo_entrada)
    audio_rapido = audio._spawn(audio.raw_data, overrides={"frame_rate": int(audio.frame_rate * velocidad)})
    audio_rapido = audio_rapido.set_frame_rate(audio.frame_rate)
    audio_rapido.export(archivo_salida, format="mp3")


def generar_voz_espanol(texto, archivo_salida="boletin_trafico.mp3", velocidad=1.25):
    texto_limpio = re.sub(r'[*#\_]', '', texto)
    partes = re.split(r'(?<=[.?!])\s+', texto_limpio)

    fragmentos = []
    for parte in partes:
        if len(parte) > 180:
            fragmentos.extend(re.split(r'(?<=[,;])\s+', parte))
        else:
            fragmentos.append(parte)

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    bytes_audio_totales = bytearray()

    for fragmento in fragmentos:
        fragmento = fragmento.strip()
        if not fragmento:
            continue
        base_url = "https://translate.google.com/translate_tts"
        params = {"ie": "UTF-8", "q": fragmento, "tl": "es", "client": "tw-ob"}
        res = requests.get(base_url, params=params, headers=headers)
        if res.status_code == 200:
            bytes_audio_totales.extend(res.content)

    if bytes_audio_totales:
        temp_file = "temp_boletin.mp3"
        with open(temp_file, "wb") as f:
            f.write(bytes_audio_totales)
        acelerar_audio(temp_file, archivo_salida, velocidad=velocidad)
        return True
    return False

## 9. Llamada a Gemini con reintentos

`generar_contenido_con_reintentos(...)` tolera el error `503 UNAVAILABLE` (modelo saturado, intermitente por parte de Google), reintentando hasta 4 veces con espera creciente (15 s, 30 s, 45 s).

In [38]:
def generar_contenido_con_reintentos(client, model, contents, intentos=4, espera_inicial=15):
    ultimo_error = None
    for intento in range(1, intentos + 1):
        try:
            return client.models.generate_content(model=model, contents=contents)
        except genai_errors.ServerError as e:
            ultimo_error = e
            print(f"Aviso: Gemini no disponible (intento {intento}/{intentos}): {e}")
            if intento < intentos:
                time.sleep(espera_inicial * intento)
    raise ultimo_error

## 10. Envío a Telegram

Construye la URL del método `sendAudio`, usa el texto del boletín como `caption` (truncado a 1024 caracteres, límite de Telegram) y envía el mp3. Aislada aquí para poder llamarla **a mano** más abajo.

In [39]:
def enviar_a_telegram(archivo_audio, texto_transcripcion):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendAudio"
    caption = f"🎙️ **Boletín de Tráfico DGT**\n\n{texto_transcripcion}"
    if len(caption) > 1024:
        caption = caption[:1020] + "..."

    with open(archivo_audio, "rb") as audio:
        payload = {"chat_id": TELEGRAM_CHAT_ID, "caption": caption, "parse_mode": "Markdown"}
        files = {"audio": audio}
        r = requests.post(url, data=payload, files=files)

    if r.status_code == 200:
        print("✅ Enviado a Telegram correctamente.")
    else:
        print(f"⚠️ Telegram devolvió un error ({r.status_code}): {r.text}")
    return r

## 11. Generar el boletín completo (sin enviarlo todavía)

Equivalente al `main()` del script, parado justo antes del envío a Telegram. Junta los datos del Ayuntamiento (calles de la ciudad) y de la DGT (carreteras), y se los pasa a Gemini en un único prompt que le indica: resumir las calles de Zaragoza justo después del saludo (o saltárselo en silencio si no hay datos), luego el tiempo, luego el detalle ampliado de la provincia de Zaragoza, y por último el resto de zonas más resumido.

In [40]:

def generar_boletin_completo():
    datos_trafico = obtener_incidencias_texto()
    if "Sin incidencias" in datos_trafico or "Error" in datos_trafico:
        print(f"No hay boletín que generar: {datos_trafico}")
        return None, None

    datos_ciudad_zaragoza = obtener_incidencias_calles_zaragoza()

    saludo_dinamico = obtener_saludo_y_momento()
    fecha_hora_str = obtener_fecha_hora_generacion()

    client = genai.Client(api_key=GEMINI_API_KEY)

    prompt = f"""
Eres un locutor de radio experto en información de tráfico y tiempo regional. Genera un boletín locutado fluido (240-300 palabras) para ser leído en voz alta.

ORDEN SECUENCIAL OBLIGATORIO DE LA LOCUCIÓN:

1. INTRODUCCIÓN Y TIEMPO (Obligatorio al inicio):
   - Empieza con el saludo exacto "{saludo_dinamico}".
   - Añade a continuación la fecha y la hora exactamente así: "son las {fecha_hora_str}".
   - Incluye inmediatamente un apunte meteorológico rápido (10-15 palabras) sobre la situación del tiempo en el valle del Ebro y la zona norte.

2. INCIDENCIAS URBANAS EN ZARAGOZA CIUDAD (Ayuntamiento):
   - Justo después de la meteorología, pasa a informar sobre las calles de la ciudad de Zaragoza.
   - Revisa TODAS las incidencias urbanas proporcionadas en "DATOS AYUNTAMIENTO DE ZARAGOZA".
   - Ofrece un panorama representativo: menciona los cortes y obras más destacados por el nombre exacto de la calle o avenida que aparece en los datos (por ejemplo, afecciones en Dr. Blanco Cordero, Marina Española por las obras del Río Huerva, Avenida Valencia, Coso, etc.), sin omitir puntos relevantes. Si hay obras menores secundarias, puedes agruparlas de forma natural.
   - Si el texto de esos datos indica que no hay incidencias o que hubo un error, salta a la siguiente sección en silencio sin mencionarlo.

3. CARRETERAS INTERURBANAS DE DGT (Zaragoza provincia y resto):
   - Dedica la parte principal de la información interurbana a las carreteras de la provincia de Zaragoza. Incluye TODAS las incidencias (retenciones, obras, obstáculos) indicando sentido, carril o punto kilométrico cuando existan en los datos.
   - Termina con un breve repaso ágil a las carreteras del resto de zonas (Huesca, Teruel, Navarra y La Rioja).

REGLAS DE FORMATO:
- NO uses emojis, asteriscos (*) ni encabezados markdown (##).
- Redacta con estilo de radio: lenguaje natural, articulado y fluido para locución en directo.

DATOS AYUNTAMIENTO DE ZARAGOZA (calles de la ciudad):
{datos_ciudad_zaragoza}

DATOS DGT (carreteras interurbanas):
{datos_trafico}
"""

    try:
        response = generar_contenido_con_reintentos(client, model='gemini-3.6-flash', contents=prompt)
    except genai_errors.ServerError as e:
        print(f"Gemini siguió sin estar disponible tras varios reintentos: {e}")
        return None, None

    texto_informe = response.text.strip()

    archivo_mp3 = "boletin_trafico.mp3"
    if generar_voz_espanol(texto_informe, archivo_mp3, velocidad=1.25):
        print("✅ Boletín generado. Revísalo abajo antes de enviarlo.")
        return archivo_mp3, texto_informe

    print("⚠️ No se pudo generar el audio.")
    return None, texto_informe

## 12. Ejecutar la generación

Ejecuta esta celda cada vez que quieras un boletín nuevo. Guarda el resultado en `archivo_mp3_generado` y `texto_informe_generado`, que usará la celda de envío más abajo.

In [41]:
archivo_mp3_generado, texto_informe_generado = generar_boletin_completo()
print(texto_informe_generado)

✅ Boletín generado. Revísalo abajo antes de enviarlo.
Boletín de tráfico actualizado el lunes 21 de septiembre, 00:15 horas. Buenas noches. Comenzamos en las calles de Zaragoza, donde las obras condicionan el tráfico con cortes y desvíos en la Avenida de Valencia, el Coso, la Avenida de Madrid y la calle Anselmo Clavé, además de trabajos en Pedro Cerbuna y Marqués de la Cadena. En lo meteorológico, disponemos de cielos nubosos y ambiente fresco en todo el valle del Ebro.

En cuanto a la provincia de Zaragoza, mucha precaución en la Autovía del Ebro, la A-68, a la altura de El Burgo de Ebro, con el carril izquierdo cerrado por obras entre los kilómetros 224 y 225 en sentido decreciente. También por obras hay tráfico alternativo en la N-232, entre los kilómetros 213 y 220, entre El Burgo y Fuentes de Ebro, afectando a ambos sentidos y a todos los carriles. Además, en la Autovía del Nordeste, la A-2, un vehículo atascado dificulta el paso a la altura del kilómetro 317 en sentido creciente

### (Opcional) Escuchar el audio generado antes de enviarlo

In [34]:
if archivo_mp3_generado:
    display(Audio(archivo_mp3_generado))
else:
    print("No hay audio generado todavía. Ejecuta la celda de generación primero.")

## 13. Envío manual a Telegram

Esta es la parte que en el repo ahora corre sola dentro de la GitHub Action. Aquí la ejecutas tú, cuando quieras, sobre el boletín que acabas de generar y revisar arriba. No hace falta volver a generar nada: reutiliza `archivo_mp3_generado` y `texto_informe_generado`.

In [35]:
if archivo_mp3_generado:
    enviar_a_telegram(archivo_mp3_generado, texto_informe_generado)
else:
    print("No hay nada que enviar: genera el boletín primero (celda de la sección 12).")

✅ Enviado a Telegram correctamente.
